In [1]:
import pandas as pd
import json
import os

In [5]:
def load_and_preprocess_data(file_path, label):
    """JSON 파일을 로드하고 전처리하여 DataFrame으로 반환"""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 중첩된 JSON을 평탄화
    df = pd.json_normalize(data)

    # 타겟 변수 추가 (집중: 1, 비집중: 0)
    df['target'] = label

    return df

In [6]:
focused_file = '../data/집중.json'
unfocused_file = '../data/비집중.json'

# 파일 존재 여부 확인
if not os.path.exists(focused_file):
    # 대체 경로 시도
    focused_file = './data/집중.json'
    if not os.path.exists(focused_file):
        focused_file = '/data/집중.json'

if not os.path.exists(unfocused_file):
    # 대체 경로 시도
    unfocused_file = './data/비집중.json'
    if not os.path.exists(unfocused_file):
        unfocused_file = '/data/비집중.json'

print(f"집중 파일 경로: {focused_file}")
print(f"비집중 파일 경로: {unfocused_file}")

# 데이터 로드
df_focused = load_and_preprocess_data(focused_file, 1)
df_unfocused = load_and_preprocess_data(unfocused_file, 0)

# 두 데이터프레임을 하나로 합치기
df = pd.concat([df_focused, df_unfocused], ignore_index=True)

# 데이터 확인
print(f"데이터 로드 완료: {len(df)} 행")
print("데이터프레임 컬럼:", df.columns.tolist())  # 컬럼명 확인
df.head()


집중 파일 경로: ../data/집중.json
비집중 파일 경로: ../data/비집중.json
데이터 로드 완료: 1440 행
데이터프레임 컬럼: ['timestamp', 'eye_status.status', 'eye_status.ear_value', 'head_pose.pitch', 'head_pose.yaw', 'head_pose.roll', 'target']


,timestamp,eye_status.status,eye_status.ear_value,head_pose.pitch,head_pose.yaw,head_pose.roll,target
0,1754727463713,NO_FACE_DETECTED,0.0000,0.00,0.00,0.00,1
1,1754727463955,OPEN,0.4369,12.47,26.35,18.40,1
2,1754727464140,OPEN,0.4446,6.59,42.88,97.74,1
3,1754727464344,OPEN,0.4560,4.49,-9.02,-39.15,1
4,1754727464487,OPEN,0.4586,2.99,-34.97,-107.48,1


In [11]:
from pycaret.classification import setup, compare_models, plot_model, save_model, predict_model

# PyCaret 환경 설정 - 분할 파라미터 명시적 지정
clf_setup = setup(data=df,
                  target='target',
                  session_id=123,
                  train_size=0.5,
                  fold=5,                 # 5-fold 교차 검증
                  fold_strategy='stratifiedkfold',  # 계층적 교차 검증 (올바른 파라미터 값)
                  categorical_features=['eye_status.status'],
                  verbose=True)           # 세부 정보 출력

# 여러 모델을 학습하고 성능 비교
best_model = compare_models(fold=5)  # 5-fold 교차 검증으로 비교

print("\n3. 최적 모델 분석 및 저장을 시작합니다...")
print("가장 성능이 좋은 모델:")
print(best_model)

# 테스트 데이터에서의 성능 평가
test_predictions = predict_model(best_model)
print("\n테스트 데이터에서의 성능:")
print(test_predictions.head())

# 평가 결과를 이미지 파일로 저장
try:
    plot_model(best_model, plot='confusion_matrix', save=True)
    plot_model(best_model, plot='auc', save=True)

    # 모델이 feature_importances_나 coef_를 지원하는지 확인
    if hasattr(best_model, 'feature_importances_') or hasattr(best_model, 'coef_'):
        plot_model(best_model, plot='feature', save=True)
        print("특성 중요도 플롯이 저장되었습니다.")
    else:
        print("선택된 모델은 특성 중요도 시각화를 지원하지 않습니다.")

    print("\n평가 결과가 이미지 파일로 저장되었습니다.")
except Exception as e:
    print(f"\n평가 결과 이미지 저장 중 오류 발생: {e}")

# 모델 저장
save_model(best_model, '../app/models/checkpoints/concentration_model')
print("\n학습된 모델이 'concentration_model.pkl' 파일로 저장되었습니다.")

,Description,Value
0,Session id,123
1,Target,target
2,Target type,Binary
3,Original data shape,"(1440, 7)"
4,Transformed data shape,"(1440, 9)"
5,Transformed train set shape,"(720, 9)"
6,Transformed test set shape,"(720, 9)"
7,Numeric features,5
8,Categorical features,1
9,Preprocess,True


,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,20:38:21
Status,. . . . . . . . . . . . . . . . . .,Loading Dependencies
Estimator,. . . . . . . . . . . . . . . . . .,Compiling Library


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
knn,K Neighbors Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.2520
nb,Naive Bayes,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0420
ridge,Ridge Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0280
lda,Linear Discriminant Analysis,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.3440
lightgbm,Light Gradient Boosting Machine,0.9986,1.0000,0.9950,1.0000,0.9975,0.9965,0.9965,0.3120
qda,Quadratic Discriminant Analysis,0.7417,0.9324,0.7600,0.7123,0.5948,0.4597,0.5102,0.0400
lr,Logistic Regression,0.7222,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.7940
dummy,Dummy Classifier,0.7222,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0300
svm,SVM - Linear Kernel,0.6333,0.8000,0.2000,0.0556,0.0870,0.0000,0.0000,0.0220



3. 최적 모델 분석 및 저장을 시작합니다...
가장 성능이 좋은 모델:
KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                     metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                     weights='uniform')


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,K Neighbors Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000



테스트 데이터에서의 성능:
          timestamp eye_status.status  eye_status.ear_value  head_pose.pitch  \
586   1754728037611              OPEN                0.4267         6.490000   
1385  1754728198598              OPEN                0.4203        -0.830000   
739   1754728068206              OPEN                0.3101        -1.660000   
594   1754728039234              OPEN                0.4590       -31.540001   
889   1754728098214            CLOSED                0.1622        -4.690000   

      head_pose.yaw  head_pose.roll  target  prediction_label  \
586      -60.860001       -5.820000       0                 0   
1385      70.080002      175.240005       0                 0   
739      -68.699997     -156.669998       0                 0   
594      -69.510002     -132.309998       0                 0   
889       -0.140000      -40.090000       0                 0   

      prediction_score  
586                1.0  
1385               1.0  
739                1.0  
594         

선택된 모델은 특성 중요도 시각화를 지원하지 않습니다.

평가 결과가 이미지 파일로 저장되었습니다.
Transformation Pipeline and Model Successfully Saved

학습된 모델이 'concentration_model.pkl' 파일로 저장되었습니다.
